# 1. Extracción, Enriquecimiento e Inteligencia Científica de Aditivos

Este proyecto implementa un flujo de trabajo avanzado de **Ingeniería de Datos y Minería de Textos Científicos** para transformar una lista básica de aditivos alimentarios en un dataset multidimensional. El objetivo es cuantificar la evidencia científica asociada a riesgos y beneficios para la salud mediante el uso de APIs externas.

---

##  Fases del Proceso

### 1. Extracción de Taxonomía (Open Food Facts)
La base del proyecto es la obtención de datos normalizados sobre sustancias alimentarias.
* **Conexión a la API**: Se accede a la taxonomía oficial de *Open Food Facts* para obtener un catálogo global de aditivos.
* **Filtrado mediante Regex**: La función `generar_maestro_aditivos_raw` identifica códigos que cumplen exclusivamente el estándar europeo (ej. "E300" para ácido ascórbico).
* **Gestión de descartes**: Se auditan los elementos que no cumplen el patrón (nombres químicos sin código E o categorías generales) para asegurar la integridad del "maestro" de aditivos.

In [ ]:
import numpy as np
import os
import pandas as pd
import pubchempy as pcp
import re
import requests
import time
import urllib.parse
from tqdm import tqdm

In [3]:
def generar_maestro_aditivos_raw():
    
    """
    Descarga y procesa la taxonomía de aditivos de Open Food Facts para generar un
    catálogo de códigos E (aditivos alimentarios aceptados).

    La función se conecta a la API pública de Open Food Facts, obtiene el diccionario
    completo de aditivos y filtra solo aquellos códigos que cumplen con el patrón
    estándar de los aditivos europeos (por ejemplo, 'e300' para ácido ascórbico).

    """
    
    url = "https://world.openfoodfacts.org/data/taxonomies/additives.json"
    headers = {'User-Agent': 'MyDataScienceProject/1.0 (mklanibarro@gmail.com)'}
    
    print("Conectando con Open Food Facts...")
    try:
        response = requests.get(url, headers=headers, timeout=30)
        if response.status_code == 200:
            data = response.json()
            lista_limpia = []
            diccionario_descartes = {}

            # Regla para Regex: Empieza por 'e', seguido de 1 a 7 caracteres alfanuméricos
            patron = re.compile(r'^e[0-9a-z]{1,7}$', re.IGNORECASE)

            for key, value in data.items():
                # Extraemos el ID corto (ej: de 'en:e300' a 'e300')
                id_corto = key.split(':')[-1] if ':' in key else key
                nombre_en = value.get('name', {}).get('en', 'Desconocido')

                if patron.match(id_corto):
                    lista_limpia.append({
                        'id': key,
                        'codigo_e': id_corto.upper(),
                        'nombre': nombre_en
                    })
                else:
                    # Todo lo que no cumpla la regla va al diccionario de descartes
                    diccionario_descartes[key] = nombre_en
            
            # Crear DataFrame y guardar
            df = pd.DataFrame(lista_limpia)
            df.to_csv('../data/maestro_aditivos_raw.csv', index=False)
            
            print(f"✅ ¡Hecho! Archivo 'maestro_aditivos_raw.csv' guardado.")
            print(f"📊 Aditivos válidos: {len(df)}")
            print(f"📦 Elementos descartados: {len(diccionario_descartes)}")
            
            return df, diccionario_descartes
        else:
            print(f"❌ Error de conexión: Código {response.status_code}")
    except Exception as e:
        print(f"❌ Error inesperado: {e}")
    
    return None, None

# Ejecución
df_final, descartes = generar_maestro_aditivos_raw()

Conectando con Open Food Facts...
✅ ¡Hecho! Archivo 'maestro_aditivos_raw.csv' guardado.
📊 Aditivos válidos: 651
📦 Elementos descartados: 30


### 2. Limpieza y Normalización
Refinamiento de los datos crudos para garantizar búsquedas precisas en bases de datos externas.
* **Estandarización**: Conversión de los códigos `codigo_e` a mayúsculas y limpieza de caracteres especiales.
* **Refinamiento de Nombres**: Separación de códigos y nombres descriptivos para obtener términos químicos limpios (ej. extraer "Dicalcium citrate" de cadenas complejas).

In [ ]:
"""
AUDITORÍA DE DESCARTES:
1. Estructura los datos descartados en un DataFrame para facilitar su análisis.
2. Inspecciona visualmente los primeros registros para detectar patrones de ruido.
3. Filtra nombres científicos específicos que podrían ser aditivos válidos sin código E 
   identificado, asegurando que no se pierda evidencia científica relevante.
"""

# 1. Convertir a DataFrame para poder filtrar
df_descartes = pd.DataFrame(list(descartes.items()), columns=['ID_OFF', 'Nombre_Cientifico'])

# 2. Visualización rápida de los primeros 30
print("--- Primeros elementos descartados ---")
display(df_descartes.head(30))

# 3. Búsqueda de posibles 'Falsos Negativos' 
# (Aditivos reales que no tienen código E pero sí nombre químico)
falsos_negativos = df_descartes[df_descartes['Nombre_Cientifico'].str.contains('acid|carbonate|phosphate', case=False, na=False)]
print("\n--- Posibles aditivos detectados por nombre químico en los descartes ---")
display(falsos_negativos.head(20))

--- Primeros elementos descartados ---


,ID_OFF,Nombre_Cientifico
0,en:methylsulfonylmethane,Methylsulfonylmethane - dimethyl sulfone
1,en:no9,No9 - n9
2,en:no8,No8 - n8
3,en:organic-acids,Organic acids
4,en:fd-c,FD&C - FD and C
5,en:no4,No4 - n4
6,de:natürliche-quellkohlensäure,Desconocido
7,en:no6,No6 - n6
8,en:amino-acids,Amino acids
9,en:n,N° - no



--- Posibles aditivos detectados por nombre químico en los descartes ---


,ID_OFF,Nombre_Cientifico
3,en:organic-acids,Organic acids
8,en:amino-acids,Amino acids
11,en:diphosphates-and-sodium-carbonates,Diphosphates and sodium carbonates
12,en:di-and-triphosphates,Di- and triphosphates
13,en:nucleic-acids,Nucleic acids
22,en:mono-and-diglycerides-of-vegetable-fatty-acids,Mono- and diglycerides of vegetable fatty acids
28,en:ammonium-and-sodium-carbonates,Ammonium and sodium carbonates


El descarte de aditivos ha sido exitoso ya que se han descartado grupos genéricos o confusos

In [4]:
"""
LIMPIEZA Y ESTANDARIZACIÓN:
1. Formatea la columna 'codigo_e' eliminando prefijos de idioma (en:, fr:) y normalizando a mayúsculas.
2. Refina la nomenclatura eliminando códigos repetidos dentro del nombre para dejar solo el término químico.
3. Exporta un archivo maestro depurado que servirá como base única de verdad para el enriquecimiento.
"""

# Cargar el CSV sucio
df_maestro = pd.read_csv('../data/maestro_aditivos_raw.csv')

# 1. Extraer el código E (E250, E330...)
# Suponiendo que el formato es "en:e250" en la columna 'id'
df_maestro['codigo_e'] = df_maestro['id'].str.split(':').str[-1].str.upper()

# 2. Limpiar la columna nombre
# Si el nombre viene como "E333ii - Dicalcium citrate", cortamos por el guion
df_maestro['nombre_limpio'] = df_maestro['nombre'].str.split(' - ').str[-1]

# 3. Guardar el maestro reluciente
df_maestro[['id', 'codigo_e', 'nombre_limpio']].to_csv('../data/maestro_aditivos_limpio.csv', index=False)

### 3. Enriquecimiento Químico (PubChem)
Se integra la API de **PubChem** para ampliar la capacidad de detección del modelo y normalizar la nomenclatura química.
* **Sistema de Fallback (Redundancia)**: La función `enriquecer_con_pubchem` intenta localizar el compuesto primero por su nombre; si falla, utiliza el código E con contextos adicionales para evitar ambigüedades.
* **Identificadores y Sinónimos**: Se recupera el **CID** (Compound ID) y una lista de sinónimos relevantes. Esto permite que las búsquedas bibliográficas posteriores sean mucho más exhaustivas al incluir nombres alternativos.

In [ ]:
# 1. Cargar tu archivo maestro
df_maestro = pd.read_csv('../data/maestro_aditivos_limpio.csv')

# 2. Función de búsqueda con lógica de redundancia (Fallback)

def enriquecer_con_pubchem(row):
    
    """
FASE 3: ENRIQUECIMIENTO MOLECULAR Y NORMALIZACIÓN QUÍMICA (PUBCHEM)

Esta sección actúa como un puente entre la taxonomía de Open Food Facts y la base de datos molecular PubChem.
Su objetivo es estandarizar la identidad de cada aditivo para optimizar la minería de textos posterior.

LÓGICA DE PROCESAMIENTO:
1. Sistema de Redundancia (Fallback): La función prioriza la búsqueda por el nombre químico limpio. 
   Si falla, utiliza el código E. Si el código es numérico (ej. 440), reintenta automáticamente 
   añadiendo el prefijo "E" para proporcionar contexto químico a la API.

2. Obtención de Identificadores (CID): Recupera el PubChem Compound ID, un identificador universal 
   que garantiza que cada aditivo sea tratado como una entidad química única e inequívoca.

3. Generación de Tesauro de Sinónimos: Extrae los 5 sinónimos más relevantes de PubChem. 
   Esto es crítico para la fase de PubMed, ya que permite buscar evidencia científica utilizando 
   nomenclatura técnica, comercial y códigos internacionales simultáneamente.

4. Ingeniería de Control: Implementa un límite de tasa (0.2s de espera) para cumplir con las 
   políticas de uso de la NLM y utiliza barras de progreso (tqdm) para monitorizar el 
   enriquecimiento de los ~650 registros en tiempo real.
"""
    
    nombre = row.get('nombre')
    codigo = row.get('codigo_e')
    
    # Lógica de decisión: ¿Qué usamos para buscar?
    if pd.notna(nombre) and str(nombre).strip() != "":
        termino_busqueda = str(nombre).strip()
    elif pd.notna(codigo) and str(codigo).strip() != "":
        termino_busqueda = str(codigo).strip()
    else:
        return None, None # Si no hay ninguno de los dos, saltamos
    
    try:
        # Buscamos en PubChem
        results = pcp.get_compounds(termino_busqueda, 'name')
        
        # Si no encuentra nada con el código a secas (ej: "440"), 
        # reintentamos añadiendo "E" o "additive" para dar contexto
        if not results and termino_busqueda == str(codigo).strip():
            results = pcp.get_compounds(f"E{termino_busqueda}", 'name')
            
        if results:
            compuesto = results[0]
            cid = compuesto.cid
            synonyms_list = pcp.get_synonyms(cid)
            all_syns = synonyms_list[0]['Synonym'] if synonyms_list else []
            top_syns = "|".join(all_syns[:5])
            return cid, top_syns
            
    except Exception:
        return None, None
    
    return None, None

# 3. Bucle de procesamiento
cids = []
sinonimos = []

print("Iniciando búsqueda en PubChem con sistema de redundancia...")
for index, row in tqdm(df_maestro.iterrows(), total=df_maestro.shape[0]):
    cid, syns = enriquecer_con_pubchem(row)
    cids.append(cid)
    sinonimos.append(syns)
    time.sleep(0.2) # Respetar el límite de la API

# 4. Guardado final
df_maestro['pubchem_cid'] = cids
df_maestro['pubchem_synonyms'] = sinonimos

df_maestro.to_csv('../data/maestro_aditivos_enriquecido.csv', index=False)
print("\n¡Listo! El CSV ha sido enriquecido incluso para aditivos sin nombre.")

Iniciando búsqueda en PubChem con sistema de redundancia...


100%|██████████| 651/651 [12:04<00:00,  1.11s/it]


¡Listo! El CSV ha sido enriquecido incluso para aditivos sin nombre.


### 4. Minería de Evidencia Científica (PubMed)
Fase final de generación de métricas de investigación basadas en la base de datos médica **PubMed**.
* **Dimensiones de Análisis**: Se definen **29 categorías críticas**, cubriendo desde toxicidad genética y carcinogenicidad hasta la nueva frontera de la microbiota (disbiosis intestinal).
* **Consultas Automatizadas**: El script construye *queries* complejas uniendo: `(Nombre OR Código E OR Sinónimo) AND (Términos MeSH específicos del Riesgo)`.
* **Resiliencia del Pipeline**: Implementación de un sistema de **Auto-Resume** para retomar el proceso en caso de interrupción y gestión estricta de límites de tasa (*Rate Limiting*) de la API de la NCBI.

In [ ]:
"""
SISTEMA DE MINERÍA DE TEXTO CIENTÍFICO Y METRIZACIÓN TOXICOLÓGICA (PUBMED)

Este módulo constituye el núcleo analítico del proyecto. Su función es cuantificar la 
evidencia científica global para cada aditivo, transformando términos cualitativos 
en métricas cuantitativas distribuidas en 26 dimensiones de riesgo y salud.

COMPONENTES PRINCIPALES:

1. CONFIGURACIÓN Y CONTROL DE API:
   - Implementa credenciales y parámetros de entorno para la API E-utils de la NCBI.
   - Establece un 'LIMITE_CAPPING' (200.000) para normalizar valores extremos y 
     optimizar la escala de los datos para futuros modelos de Machine Learning.

2. DICCIONARIO SEMÁNTICO DE BÚSQUEDA (26 Dimensiones):
   - Estructura consultas complejas utilizando operadores booleanos y términos MeSH 
     (Medical Subject Headings).
   - Clasifica la evidencia en: Riesgos Celulares, Sistémicos, Impacto en Microbiota, 
     Respuesta Inmune, Bioactividad Positiva y Toxicidad específica por familias.

3. MOTOR DE CONSULTA RESILIENTE (Lógica de Limpieza y Redundancia):
   - 'limpiar_termino': Normaliza los nombres para evitar errores sintácticos en PubMed.
   - 'construir_query_nombres': Genera una base de búsqueda robusta que une el nombre 
     limpio, el código E (con variaciones numéricas) y sinónimos de PubChem para 
     maximizar la exhaustividad de los resultados.
   - 'query_pubmed_count': Gestiona la comunicación HTTP con lógica de reintento 
     automático ante errores de red o límites de tasa (Errores 429/500).

4. PIPELINE DE EJECUCIÓN CON AUTO-RESUME:
   - El proceso está diseñado para ser interrumpible; detecta si existe un archivo 
     de salida previo y retoma la minería desde el último ID procesado.
   - Implementa un 'Time Sleep' dinámico (0.12s) para cumplir estrictamente con 
     el límite de 10 peticiones por segundo de PubMed.
   - Realiza guardados incrementales cada 5 registros para prevenir la pérdida 
     de datos ante fallos inesperados o interrupciones manuales.
"""

# ==============================================================================
# 1. CONFIGURACIÓN
# ==============================================================================
MI_EMAIL = "mklanibarro@gmail.com"
NOMBRE_HERRAMIENTA = "AdvancedFoodAdditiveResearch_DS"
API_KEY = "2ffe948859f4629dcc0057ed52ab3ca8cc09"  # ¡Cámbiala por tu clave real!

ARCHIVO_MAESTRO = '../data/maestro_aditivos_enriquecido.csv'
ARCHIVO_SALIDA = '../data/dataset_final_aditivos_24D.csv'

LIMITE_CAPPING = 200000
COL_NOMBRE = 'nombre_limpio'
COL_CODIGO = 'codigo_e'
COL_SINONIMOS = 'pubchem_synonyms'

# ==============================================================================
# 2. DICCIONARIO COMPLETO (26 dimensiones)
# ==============================================================================
DICCIONARIO_BUSQUEDA = {
    # ==========================================================================
    # 1. RIESGOS ESTRUCTURALES Y CELULARES
    # ==========================================================================
    "toxicidad_genetica": (
        '("DNA damage"[Mesh] OR genotoxicity OR genotoxic OR mutagenicity OR mutagenic '
        'OR "Ames test" OR "comet assay" OR "micronucleus test" OR "chromosomal aberration" '
        'OR "sister chromatid exchange" OR "gene mutation" OR clastogenicity OR aneugenicity)'
    ),
    "carcinogenesis": (
        '("carcinogenesis"[Mesh] OR carcinogenicity OR carcinogenic OR "tumor induction" OR oncogenic '
        'OR "neoplasm"[Mesh] OR "malignant transformation" OR "cancer promotion" OR "cancer progression")'
    ),
    "estres_oxidativo": (  # Nueva categoría crítica
        '("oxidative stress"[Mesh] OR "reactive oxygen species"[Mesh] OR ROS OR "lipid peroxidation" '
        'OR "glutathione depletion" OR "superoxide dismutase" OR catalase OR "oxidative damage")'
    ),
    
    # ==========================================================================
    # 2. RIESGOS SISTÉMICOS (endocrino, metabolismo, hígado, riñón, corazón, nervioso)
    # ==========================================================================
    "disrupcion_endocrina": (
        '("endocrine disruptors"[Mesh] OR "endocrine disruption" OR "estrogenic activity" OR xenoestrogen '
        'OR "anti-androgenic" OR "thyroid peroxidase inhibition" OR "thyroid hormone disruption" '
        'OR "estrogen receptor agonist" OR "androgen receptor antagonist" OR "reproductive toxicity"[Mesh])'
    ),
    "toxicidad_metabolica": (
        '("metabolic syndrome"[Mesh] OR "insulin resistance"[Mesh] OR obesity OR diabetogenic '
        'OR "glucose intolerance" OR hyperglycemia OR dyslipidemia OR adipogenesis OR "fatty liver" OR steatosis)'
    ),
    "hepatotoxicidad": (
        '("chemical and drug induced liver injury"[Mesh] OR hepatotoxicity OR "liver damage" '
        'OR "elevated liver enzymes" OR "hepatic steatosis" OR "liver fibrosis")'
    ),
    "nefrotoxicidad": (
        '(nephrotoxicity OR "kidney injury" OR "renal damage" OR "tubular necrosis" OR proteinuria OR nephritis)'
    ),
    "cardiotoxicidad": (
        '(cardiotoxicity OR "myocardial damage" OR arrhythmia OR "QT prolongation" OR "cardiac hypertrophy" '
        'OR cardiomyopathy OR "endothelial dysfunction")'
    ),
    "neurotoxicidad": (
        '("neurotoxicity"[Mesh] OR neurotoxic OR "neuron damage" OR demyelination OR neuroinflammation '
        'OR "cognitive deficit" OR "memory impairment" OR "dopaminergic toxicity" OR excitotoxicity)'
    ),
    
    # ==========================================================================
    # 3. BARRERAS Y MICROBIOTA (La nueva frontera)
    # ==========================================================================
    "microbiota_dysbiosis": (
        '("dysbiosis"[Mesh] OR "gastrointestinal microbiome"[Mesh] OR "leaky gut" OR "intestinal permeability" '
        'OR "tight junction disruption" OR "LPS translocation" OR "butyrate reduction" '
        'NOT (probiotic OR prebiotic OR "fecal transplant"))'
    ),
    "barrera_hematoencefalica": (
        '("blood-brain barrier"[Mesh] OR "BBB permeability" OR neuroinflammation OR "astrocyte activation" '
        'OR "microglial activation" OR "cognitive impairment")'
    ),
    "barrera_intestinal": (  # Complementaria
        '("intestinal barrier function" OR "mucus layer" OR "goblet cell" OR zonulin OR occludin OR claudin)'
    ),
    
    # ==========================================================================
    # 4. RESPUESTA INMUNE (pro-inflamación, alergia, autoinmunidad)
    # ==========================================================================
    "inflamacion_pro": (
        '("inflammation"[Mesh] OR "pro-inflammatory" OR "NF-kappa B"[Mesh] OR "cytokine storm" '
        'OR "IL-1 beta" OR "IL-6" OR "TNF-alpha" OR "macrophage activation" OR "acute phase response" OR CRP '
        'NOT ("anti-inflammatory" OR "IL-10" OR "TGF-beta"))'
    ),
    "alergia_inmune": (
        '("hypersensitivity"[Mesh] OR anaphylaxis OR allergenicity OR urticaria OR "food allergy" '
        'OR "mast cell degranulation" OR "histamine release" OR IgE OR "contact dermatitis")'
    ),
    "autoinmunidad": (  # Nueva
        '("autoimmunity"[Mesh] OR "autoimmune disease" OR "systemic lupus erythematosus" OR "rheumatoid arthritis" '
        'OR autoantibodies OR "molecular mimicry" OR "adjuvant activity")'
    ),
    
    # ==========================================================================
    # 5. BENEFICIOS Y BIOACTIVIDAD (segmentación)
    # ==========================================================================
    "antioxidante_tecnico": (
        '("antioxidants"[Mesh] OR "lipid peroxidation inhibition" OR "radical scavenging" OR "food preservation" '
        'OR "shelf life" NOT ("clinical trial"[Publication Type] OR "in vivo"))'
    ),
    "valor_prebiotico": (
        '("prebiotics"[Mesh] OR "gut health" OR "probiotic"[Mesh] OR bifidogenic OR "lactobacilli growth" '
        'OR "short-chain fatty acids" OR SCFA OR "butyrate production")'
    ),
    "citoproteccion_salud": (
        '(cytoprotective OR neuroprotective OR cardioprotective OR hepatoprotective OR gastroprotective '
        'OR hormesis OR "adaptive response" OR "healthspan")'
    ),
    "actividad_antimicrobiana": (  # Para conservadores
        '("anti-bacterial agents"[Mesh] OR antimicrobial OR antifungal OR "preservative efficacy" '
        'OR "pathogen growth inhibition" OR bacteriostatic)'
    ),
    
    # ==========================================================================
    # 6. EPIGENÉTICA Y TOXICOCINÉTICA (emergente)
    # ==========================================================================
    "epigenetica": (
        '("epigenomics"[Mesh] OR epigenetics OR "DNA methylation" OR "histone modification" OR acetylation '
        'OR "chromatin remodeling" OR "non-coding RNA" OR "miRNA expression")'
    ),
    "bioacumulacion_persistencia": (
        '("bioaccumulation"[Mesh] OR "tissue distribution" OR "organ accumulation" OR "persistent organic pollutant" '
        'OR "half-life" OR "adipose storage")'
    ),
    "nanoparticulas": (  # Clave para TiO2, sílice, etc.
        '("nanoparticles"[Mesh] OR nanomaterial OR "particle size" OR "ultrafine particles" OR translocation '
        'OR "Peyer\'s patches" OR "lymphatic transport")'
    ),
    
    # ==========================================================================
    # 7. EVIDENCIA Y METODOLOGÍA
    # ==========================================================================
    "evidencia_humana": (
        '("clinical trial"[Publication Type] OR "meta-analysis"[Publication Type] OR "randomized controlled trial"[Publication Type] '
        'OR "systematic review"[Publication Type] OR "cohort study" OR "case-control study" OR epidemiological)'
    ),
    "estudios_mecanisticos": (
        '("in vitro"[Mesh] OR "cell culture" OR "molecular mechanism" OR "signaling pathway" '
        'OR "receptor binding" OR "enzyme inhibition" OR "gene expression" OR transcriptomics OR proteomics)'
    ),
    "estudios_in_vivo": (
        '("in vivo" OR "animal model" OR mouse OR rat OR murine OR rodent OR zebrafish OR drosophila OR swine)'
    ),
    
    # ==========================================================================
    # 8. INTERACCIONES CON FÁRMACOS O NUTRIENTES (Muy relevante)
    # ==========================================================================
    "interacciones": (
        '("drug interactions"[Mesh] OR "food-drug interaction" OR "nutrient interaction" OR "synergistic toxicity" '
        'OR "bioavailability alteration" OR "cytochrome P450 interaction" OR "P-glycoprotein modulation")'
    ),
    
    # ==========================================================================
    # 9. TOXICIDAD ESPECÍFICA PARA FAMILIAS DE ADITIVOS (Opcional, muy potente)
    # ==========================================================================
    # (Estos ya son subconsultas predefinidas; puedes usarlos directamente)
    "toxicidad_emulsionantes": (
        '("polysorbate 80" OR carboxymethylcellulose OR carrageenan OR "xanthan gum" OR "gum arabic") '
        'AND (dysbiosis OR inflammation OR "intestinal permeability")'
    ),
    "toxicidad_edulcorantes": (
        '(aspartame OR sucralose OR saccharin OR "acesulfame K" OR stevia) '
        'AND (glucose intolerance OR microbiome OR neurobehavioral OR carcinogenicity)'
    ),
    "toxicidad_conservadores": (
        '("sodium benzoate" OR "potassium sorbate" OR "calcium propionate" OR "sodium nitrite") '
        'AND (DNA damage OR hypersensitivity OR mitochondrial dysfunction)'
    ),
}

# ==============================================================================
# 3. FUNCIONES DE LIMPIEZA Y CONSULTA
# ==============================================================================

def limpiar_termino(termino):
    """Elimina comillas dobles y caracteres que puedan romper la sintaxis de PubMed."""
    if not isinstance(termino, str):
        return ""
    # Eliminar comillas dobles (las reemplazamos por espacio)
    termin = termino.replace('"', ' ')
    # Eliminar caracteres no imprimibles o muy raros (dejar letras, números, espacios, guiones, paréntesis)
    termin = re.sub(r'[^\w\s\-\(\)]', ' ', termin)
    # Reducir múltiples espacios a uno
    termin = re.sub(r'\s+', ' ', termin).strip()
    return termin

def query_pubmed_count(term, max_retries=3):
    """Obtiene el conteo de PubMed con reintentos y codificación robusta."""
    term_encoded = urllib.parse.quote_plus(term, safe=':"[]/()')
    url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term={term_encoded}&retmode=json&retmax=0&tool={NOMBRE_HERRAMIENTA}&email={MI_EMAIL}&api_key={API_KEY}"
    
    for intento in range(max_retries):
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 429:
                wait = 10 * (intento + 1)
                time.sleep(wait)
                continue
            if r.status_code in (500, 502, 503, 504):
                time.sleep(5 * (intento + 1))
                continue
            r.raise_for_status()
            data = r.json()
            if "esearchresult" in data and "count" in data["esearchresult"]:
                return int(data["esearchresult"]["count"])
            return 0
        except Exception:
            if intento == max_retries - 1:
                return 0
            time.sleep(2)
    return 0

def construir_query_nombres(row):
    """Construye la base de búsqueda con nombres limpios, código E y sinónimos (máx 3 términos)."""
    componentes = []
    # Nombre principal
    if pd.notna(row[COL_NOMBRE]):
        nombre_limpio = limpiar_termino(row[COL_NOMBRE])
        if nombre_limpio:
            componentes.append(f'"{nombre_limpio}"')
    # Código E
    if pd.notna(row[COL_CODIGO]):
        cod_original = limpiar_termino(str(row[COL_CODIGO]).strip())
        if cod_original:
            componentes.append(f'"{cod_original}"')
            # Extraer número para buscar también como E123 y 123
            import re
            solo_num = re.sub(r'[^0-9]', '', cod_original)
            if solo_num and solo_num != cod_original:
                componentes.append(f'"E{solo_num}"')
                componentes.append(f'"{solo_num}"')
    # Sinónimos (máximo 2, pero limitamos a 1 para no alargar)
    if pd.notna(row[COL_SINONIMOS]):
        sinonimos = str(row[COL_SINONIMOS]).split('|')
        for s in sinonimos[:1]:  # Solo 1 sinónimo para mantener URL corta
            s_clean = limpiar_termino(s)
            if len(s_clean) > 3 and not s_clean.replace('-', '').isdigit():
                componentes.append(f'"{s_clean}"')
    # Eliminar duplicados
    componentes_uniq = list(dict.fromkeys(componentes))
    if not componentes_uniq:
        return None
    return "(" + " OR ".join(componentes_uniq) + ")"

# ==============================================================================
# 4. EJECUCIÓN CON AUTO-RESUME
# ==============================================================================

df_maestro = pd.read_csv(ARCHIVO_MAESTRO)
resultados = []

if os.path.exists(ARCHIVO_SALIDA):
    try:
        df_previo = pd.read_csv(ARCHIVO_SALIDA)
        if not df_previo.empty:
            resultados = df_previo.to_dict('records')
            ultimo_id = df_previo['id_original'].max()
            df_trabajo = df_maestro[df_maestro.index > ultimo_id]
            print(f"Retomando desde el ID original: {ultimo_id + 1}")
        else:
            df_trabajo = df_maestro
    except:
        df_trabajo = df_maestro
else:
    df_trabajo = df_maestro

if len(df_trabajo) == 0:
    print("Ya no hay aditivos por procesar. Saliendo.")
    exit(0)

print(f"Aditivos por procesar: {len(df_trabajo)}")

try:
    for i, (index, row) in enumerate(tqdm(df_trabajo.iterrows(), total=len(df_trabajo))):
        query_base = construir_query_nombres(row)
        if not query_base:
            print(f"Advertencia: índice {index} sin términos válidos, omitiendo.")
            continue

        total_real = query_pubmed_count(query_base)
        total_docs = min(total_real, LIMITE_CAPPING)

        metrizacion = {
            "id_original": index,
            "nombre_referencia": row[COL_NOMBRE],
            "total_docs": total_docs
        }

        if total_real > 0:
            for col_name, query_text in DICCIONARIO_BUSQUEDA.items():
                full_query = f"{query_base} AND {query_text}"
                conteo_real = query_pubmed_count(full_query)
                metrizacion[col_name] = min(conteo_real, LIMITE_CAPPING)
                time.sleep(0.12)  # Respetar límite de 10 req/s (0.1s + margen)
        else:
            for col_name in DICCIONARIO_BUSQUEDA.keys():
                metrizacion[col_name] = 0

        resultados.append(metrizacion)

        # Guardado cada 5 registros (más seguro)
        if (i + 1) % 5 == 0:
            pd.DataFrame(resultados).to_csv(ARCHIVO_SALIDA, index=False)

except KeyboardInterrupt:
    print("\n[!] Proceso interrumpido por el usuario. Guardando progreso...")

finally:
    if resultados:
        pd.DataFrame(resultados).to_csv(ARCHIVO_SALIDA, index=False)
        print(f"\n[OK] Dataset guardado. Total registros: {len(resultados)}")

Aditivos por procesar: 651


100%|██████████| 651/651 [2:36:38<00:00, 14.44s/it]  


[OK] Dataset guardado. Total registros: 651


In [ ]:
# 1. Cargar los datasets
df_final = pd.read_csv('../data/dataset_final_aditivos_24D.csv')
df_maestro = pd.read_csv('../data/maestro_aditivos_final_vinculado.csv')

# 2. Realizar la fusión (merge)
# Queremos conservar todas las filas de df_final y añadir la columna del maestro
# Usamos 'id_original' como llave de unión.
df_fusionado = pd.merge(
    df_final, 
    df_maestro[['id_original','codigo_e', 'id']], 
    on='id_original', 
    how='left'
)

# 3. Guardar el resultado o visualizarlo
print("✅ Fusión completada con éxito.")
print(df_fusionado[['nombre_referencia', 'id_original', 'codigo_e', 'id']].head())

# Opcional: guardar en un nuevo CSV
df_fusionado.to_csv('../data/dataset_final_con_codigos.csv', index=False)

✅ Fusión completada con éxito.
          nombre_referencia  id_original codigo_e         id
0         Dicalcium citrate            0   E333II  en:e333ii
1  Butylated hydroxytoluene            1     E321    en:e321
2         Magnesium citrate            2     E345    en:e345
3       Potassium ascorbate            3     E303    en:e303
4                    lipase            4    E1104   en:e1104


> **Resultado Final**: Un dataset enriquecido (`dataset_final_aditivos_24D.csv`) donde cada aditivo cuenta con métricas numéricas de evidencia científica, permitiendo realizar análisis de **Machine Learning** (PCA y Clustering) para identificar perfiles de riesgo en la industria alimentaria.